# Tech Challenge Fase 4 — Frente de vídeo

Detecção de **útero** e **instrumentos cirúrgicos** em laparoscopia ginecológica com YOLOv8, e alerta quando há instrumento em cena sem o útero visível.

| Seção | O que faz |
|---|---|
| 1. Preparação | Instala, conecta o Drive e define os caminhos |
| 2. Dataset | Extrai o dataset e corrige o `data.yaml` |
| 3. Treino | Treina o YOLOv8 (pula se o modelo já existir no Drive) |
| 4. Avaliação do detector | Métricas e matriz de confusão no conjunto de teste |
| 5. Demonstração do alerta | Roda o alerta num vídeo e mostra os eventos |
| 6. Avaliação do alerta | Precisão e recall do alerta na validação e no teste |

**Antes de rodar:** Ambiente de execução → Alterar tipo → **T4 GPU**. No Drive, a pasta `MyDrive/TechChallenge4` precisa ter `Uterus.v4i.yolov8.zip` e `frente_video/video/alerta_video.py`.

## 1. Preparação

In [ ]:
!pip install -q ultralytics

import glob, os, shutil, sys, zipfile, yaml, cv2
from google.colab import drive
from ultralytics import YOLO
from IPython.display import Image, display

drive.mount('/content/drive')
PASTA = '/content/drive/MyDrive/TechChallenge4'
PESOS = f'{PASTA}/best_uterus_tools_v4.pt'

sys.path.insert(0, f'{PASTA}/frente_video/video')
from alerta_video import analisar_video

## 2. Dataset

Dataset **Uterus v4** (Roboflow Universe, CC BY 4.0). O `data.yaml` original aponta para `../train/images`, fora da pasta, então geramos um corrigido.

In [ ]:
with zipfile.ZipFile(f'{PASTA}/Uterus.v4i.yolov8.zip') as z:
    z.extractall('/content/dataset')

raiz = os.path.dirname(glob.glob('/content/dataset/**/data.yaml', recursive=True)[0])
classes = yaml.safe_load(open(f'{raiz}/data.yaml'))['names']   # ['Tools', 'Uterus']

DATA_YAML = f'{raiz}/data_corrigido.yaml'
yaml.safe_dump({'path': raiz, 'train': 'train/images', 'val': 'valid/images',
                'test': 'test/images', 'names': dict(enumerate(classes))}, open(DATA_YAML, 'w'))

for parte in ('train', 'valid', 'test'):
    print(parte, len(glob.glob(f'{raiz}/{parte}/images/*')), 'imagens')

## 3. Treino

Fine-tuning do YOLOv8s pré-treinado no COCO. Se o modelo já estiver no Drive, esta etapa é pulada.

In [ ]:
if not os.path.exists(PESOS):
    YOLO('yolov8s.pt').train(data=DATA_YAML, epochs=80, imgsz=416, batch=16, patience=20,
                             project='/content/runs', name='treino', seed=42)
    shutil.copy('/content/runs/treino/weights/best.pt', PESOS)

modelo = YOLO(PESOS)
print('Modelo carregado:', modelo.names)

## 4. Avaliação do detector (conjunto de teste)

O mAP é medido sem limiar. A matriz de confusão usa confiança 0,4, que é o limiar usado pelo alerta.

In [ ]:
m = modelo.val(data=DATA_YAML, split='test', imgsz=416, project='/content/runs', name='teste', exist_ok=True)
print(f'\nGeral:  mAP50 = {m.box.map50:.3f}   mAP50-95 = {m.box.map:.3f}')
for pos, cls in enumerate(m.box.ap_class_index):
    print(f'{modelo.names[int(cls)]:<7} mAP50 = {m.box.class_result(pos)[2]:.3f}')

modelo.val(data=DATA_YAML, split='test', imgsz=416, conf=0.4,
           project='/content/runs', name='teste_conf04', exist_ok=True, verbose=False)
display(Image('/content/runs/teste_conf04/confusion_matrix.png', width=550))

## 5. Demonstração do alerta

Como não há um vídeo cirúrgico ginecológico com licença aberta, montamos um vídeo com as imagens do conjunto de teste (2 s por imagem). O resultado, o relatório e o vídeo anotado são salvos em `TechChallenge4/saida_video`.

In [ ]:
def montar_video(parte):
    imagens = sorted(glob.glob(f'{raiz}/{parte}/images/*.jpg'))
    caminho = f'/content/video_{parte}.mp4'
    gravador = cv2.VideoWriter(caminho, cv2.VideoWriter_fourcc(*'mp4v'), 5, (416, 416))
    for f in imagens:
        img = cv2.resize(cv2.imread(f), (416, 416))
        for _ in range(10):          # 10 quadros a 5 fps = 2 s por imagem
            gravador.write(img)
    gravador.release()
    return caminho, imagens

VIDEO, imagens_teste = montar_video('test')
res = analisar_video(VIDEO, PESOS, saida=f'{PASTA}/saida_video', modelo=modelo)

print('ALERTA:', res['alerta'], '| NÍVEL:', res['nivel'], '|', res['motivo'])
for e in res['eventos']:
    print(f"  {e['inicio_s']} s a {e['fim_s']} s ({e['duracao_s']} s)")
    display(Image(e['quadro'], width=320))

## 6. Avaliação do alerta

Comparamos os trechos sinalizados com a anotação do dataset:
- **Precisão:** dos quadros sinalizados, quantos de fato tinham instrumento e não tinham útero.
- **Recall:** dos trechos reais de 6 s ou mais, quantos foram sinalizados.

O limiar do útero (`conf_area`) é escolhido na **validação** e só depois medido no **teste**, para o resultado final não ficar otimista.

In [ ]:
def instrumento_sem_utero(imagem):
    rotulo = imagem.replace('/images/', '/labels/').rsplit('.', 1)[0] + '.txt'
    classes = {int(l.split()[0]) for l in open(rotulo) if l.strip()}
    return 0 in classes and 1 not in classes          # 0 = Tools, 1 = Uterus

def avaliar(parte, conf_area):
    video, imagens = montar_video(parte)
    r = analisar_video(video, PESOS, saida=f'/content/aval_{parte}', modelo=modelo,
                       conf_area=conf_area, salvar_video=False)
    real = [instrumento_sem_utero(f) for f in imagens]

    longos, i = set(), 0                              # trechos reais com 3+ imagens (6 s)
    while i < len(real):
        j = i
        while j < len(real) and real[j]:
            j += 1
        if j - i >= 3:
            longos |= set(range(i, j))
        i = max(j, i + 1)

    sinalizadas = {k for e in r['eventos'] for k in range(int(e['inicio_s'] // 2), int(e['fim_s'] // 2))}
    precisao = sum(real[k] for k in sinalizadas) / len(sinalizadas) if sinalizadas else 1.0
    recall = len(sinalizadas & longos) / len(longos) if longos else 1.0
    return precisao, recall

print('VALIDAÇÃO')
for ca in (0.4, 0.5, 0.6, 0.7):
    p, r = avaliar('valid', ca)
    print(f'  conf_area = {ca}:  precisão = {p:.2f}   recall = {r:.2f}')

print('\nTESTE')
for ca in (0.4, 0.6):
    p, r = avaliar('test', ca)
    print(f'  conf_area = {ca}:  precisão = {p:.2f}   recall = {r:.2f}')